# Bivariate Bicycle Code Enlargement

Load the bivariate bicycle CSS quantum codes from `generator_matrices/bivariate_bicycle`, enlarge `hx` and `hz` into full-rank lattice generators, and write metadata-preserving `.npz` files containing `Mqq` and `Mpp`.

In [ ]:
using LatticeDecoder
using NPZ
using Random

In [ ]:
candidate_roots = unique([abspath(pwd()), abspath(joinpath(pwd(), ".."))])
root_index = findfirst(root -> isdir(joinpath(root, "generator_matrices", "bivariate_bicycle")), candidate_roots)
root_index === nothing && error("Could not find generator_matrices/bivariate_bicycle from $(pwd())")

project_root = candidate_roots[root_index]
input_dir = joinpath(project_root, "generator_matrices", "bivariate_bicycle")
output_dir = joinpath(input_dir, "expanded")

method = :heuristic
balance_weights = true
max_iters = 256
rng = MersenneTwister(1)

mkpath(output_dir)

input_dir, output_dir

In [ ]:
function bb_prime_from_name(path::AbstractString)
    match_result = match(r"_p(\d+)(?:_expanded)?\.npz$", basename(path))
    match_result === nothing && return 2
    return parse(Int, match_result.captures[1])
end

bb_files = [
    file for file in sort(readdir(input_dir; join=true))
    if endswith(file, ".npz") && !occursin("_expanded.npz", basename(file))
]

basename.(bb_files)

In [ ]:
for input_path in bb_files
    code_name = splitext(basename(input_path))[1]
    output_path = joinpath(output_dir, "$(code_name)_expanded.npz")
    p = bb_prime_from_name(input_path)

    loaded = load_sparse_quantum_code(input_path; hx_key=:hx, hz_key=:hz, balance_weights=balance_weights)
    enlarged = enlarge_css_generators(
        loaded.Hx,
        loaded.Hz;
        p=p,
        method=method,
        max_iters=max_iters,
        rng=rng,
    )

    output = Dict{String,Any}(loaded.data)
    output["Mqq"] = enlarged.Mqq
    output["Mpp"] = enlarged.Mpp
    npzwrite(output_path, output)

    output = npzread(output_path)
    println(
        basename(input_path),
        " -> ",
        basename(output_path),
        "  Mqq=",
        size(enlarged.Mqq),
        " Mpp=",
        size(enlarged.Mpp),
        " keys=",
        sort(collect(keys(output))),
    )
end

In [ ]:
expanded_files = sort(readdir(output_dir; join=true))
basename.(expanded_files)